# Vergleichs-Tool: Hybrid (Benzin+Strom) vs. Diesel 330d vs. BEV (i4/ID.7)

**Was kann ich hier einstellen?**
- Gesamt-km
- Hybrid: Benzin (l/100 km) + Strom (kWh/100 km)
- Diesel 330d: l/100 km
- BEV (i4/ID.7): kWh/100 km
- Preise: Benzin €/l, Diesel €/l, Strom €/kWh
- CO₂-Faktoren: Benzin (kg/l), Diesel (kg/l), Strommix (kg/kWh)

**Plots**
1. Äquivalenzverbrauch (kWh/100 km)  
2. CO₂ (g/km)  
3. Kosten (€/100 km)

**Standardannahmen**
- CO₂-Faktoren: Benzin 2,31 kg/l, Diesel 2,65 kg/l; Strommix 0,33 kg/kWh (330 g/kWh).
- Heizwerte: Benzin 8,9 kWh/l, Diesel 9,7 kWh/l.
- Preise (Startwerte): Benzin 1,70 €/l, Diesel 1,60 €/l, Strom 0,50 €/kWh. 
  (ADAC: Q4/2024 ca. E5 ~1,72 €/l, E10 ~1,67 €/l, Diesel ~1,59 €/l – nutze bei Bedarf genauere 12M-Durchschnitte.)

> Hinweis: ipywidgets müssen in deiner Umgebung aktiviert sein (JupyterLab: `jupyter labextension list`, klassisches Notebook: meist out-of-the-box).


In [7]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import FloatSlider, IntSlider, VBox, Layout, HTML, Output, interactive_output

slider_style = {"description_width": "180px"}
slider_layout = Layout(width="420px")

KWH_PER_L_PETROL = 8.9
KWH_PER_L_DIESEL = 9.7

def compute_totals(km_total,
                   hyb_l_per_100, hyb_kwh_per_100,
                   diesel_l_per_100,
                   bev_kwh_per_100,
                   price_petrol, price_diesel, price_kwh,
                   co2_petrol, co2_diesel, co2_grid):
    factor = km_total / 100.0
    
    # Hybrid
    hyb_l_total   = hyb_l_per_100 * factor
    hyb_kwh_total = hyb_kwh_per_100 * factor
    hyb_cost      = hyb_l_total * price_petrol + hyb_kwh_total * price_kwh
    hyb_co2_kg    = hyb_l_total * co2_petrol + hyb_kwh_total * co2_grid
    hyb_energy_kwh_per_100 = hyb_l_per_100 * KWH_PER_L_PETROL + hyb_kwh_per_100
    hyb_energy_total_kwh   = hyb_energy_kwh_per_100 * factor
    
    # Diesel
    d_l_total = diesel_l_per_100 * factor
    d_cost    = d_l_total * price_diesel
    d_co2_kg  = d_l_total * co2_diesel
    d_energy_kwh_per_100 = diesel_l_per_100 * KWH_PER_L_DIESEL
    d_energy_total_kwh   = d_energy_kwh_per_100 * factor
    
    # BEV
    bev_kwh_total = bev_kwh_per_100 * factor
    bev_cost      = bev_kwh_total * price_kwh
    bev_co2_kg    = bev_kwh_total * co2_grid
    bev_energy_kwh_per_100 = bev_kwh_per_100
    bev_energy_total_kwh   = bev_kwh_total
    
    factor_nonzero = factor if factor != 0 else 1
    df = pd.DataFrame({
        "Hybrid (Benzin+Strom)": {
            "Energie (kWh/100km)": hyb_energy_kwh_per_100,
            "Energie gesamt (kWh)": hyb_energy_total_kwh,
            "Kosten/100km (€)": hyb_cost / factor_nonzero,
            "Kosten gesamt (€)": hyb_cost,
            "CO2 (g/km)": hyb_co2_kg * 1000 / km_total if km_total else 0,
            "CO2 gesamt (kg)": hyb_co2_kg,
        },
        "Diesel 330d": {
            "Energie (kWh/100km)": d_energy_kwh_per_100,
            "Energie gesamt (kWh)": d_energy_total_kwh,
            "Kosten/100km (€)": d_cost / factor_nonzero,
            "Kosten gesamt (€)": d_cost,
            "CO2 (g/km)": d_co2_kg * 1000 / km_total if km_total else 0,
            "CO2 gesamt (kg)": d_co2_kg,
        },
        "BEV (i4/ID.7)": {
            "Energie (kWh/100km)": bev_energy_kwh_per_100,
            "Energie gesamt (kWh)": bev_energy_total_kwh,
            "Kosten/100km (€)": bev_cost / factor_nonzero,
            "Kosten gesamt (€)": bev_cost,
            "CO2 (g/km)": bev_co2_kg * 1000 / km_total if km_total else 0,
            "CO2 gesamt (kg)": bev_co2_kg,
        },
    }).T
    return df

def plots(df):
    metrics = [
        ("Energie (kWh/100km)", "kWh pro 100 km", "Äquivalenzverbrauch"),
        ("CO2 (g/km)", "g CO₂ pro km", "CO₂-Emissionen"),
        ("Kosten/100km (€)", "€ pro 100 km", "Kosten pro 100 km"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (column, ylabel, title) in zip(axes, metrics):
        vals = df[column]
        ax.bar(vals.index, vals.values)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.tick_params(axis="x", labelrotation=20)
    fig.tight_layout()
    return fig

def build_dashboard_state(km_total=19000,
                          hyb_l_per_100=5.2, hyb_kwh_per_100=8.2,
                          diesel_l_per_100=6.0,
                          bev_kwh_per_100=20.0,
                          price_petrol=1.70, price_diesel=1.60, price_kwh=0.50,
                          co2_petrol=2.31, co2_diesel=2.65, co2_grid=0.33):
    df = compute_totals(km_total,
                        hyb_l_per_100, hyb_kwh_per_100,
                        diesel_l_per_100,
                        bev_kwh_per_100,
                        price_petrol, price_diesel, price_kwh,
                        co2_petrol, co2_diesel, co2_grid)
    fig = plots(df)
    return df.round(3), fig

heading = HTML("<h3>Fahrzeugvergleich</h3>")
intro = HTML("<p>Passt die Regler an, um Verbrauch, Kosten und CO₂-Emissionen je Antrieb zu vergleichen.</p>")

km_slider = IntSlider(description="Gesamt-km", min=1000, max=100000, step=500, value=19000, style=slider_style, layout=slider_layout)
hyb_l_slider = FloatSlider(description="Hybrid Benzin l/100", min=0, max=12, step=0.1, value=5.2, readout_format=".1f", style=slider_style, layout=slider_layout)
hyb_kwh_slider = FloatSlider(description="Hybrid kWh/100", min=0, max=25, step=0.1, value=8.2, readout_format=".1f", style=slider_style, layout=slider_layout)
diesel_slider = FloatSlider(description="Diesel l/100 (330d)", min=5.0, max=7.0, step=0.1, value=6.0, readout_format=".1f", style=slider_style, layout=slider_layout)
bev_slider = FloatSlider(description="BEV kWh/100", min=14, max=28, step=0.1, value=20.0, readout_format=".1f", style=slider_style, layout=slider_layout)
price_petrol_slider = FloatSlider(description="Preis Benzin €/l", min=1.3, max=2.2, step=0.01, value=1.70, style=slider_style, layout=slider_layout)
price_diesel_slider = FloatSlider(description="Preis Diesel €/l", min=1.3, max=2.2, step=0.01, value=1.60, style=slider_style, layout=slider_layout)
price_kwh_slider = FloatSlider(description="Preis Strom €/kWh", min=0.2, max=0.8, step=0.01, value=0.50, style=slider_style, layout=slider_layout)
co2_petrol_slider = FloatSlider(description="CO2 Benzin kg/l", min=2.2, max=2.4, step=0.01, value=2.31, style=slider_style, layout=slider_layout)
co2_diesel_slider = FloatSlider(description="CO2 Diesel kg/l", min=2.5, max=2.8, step=0.01, value=2.65, style=slider_style, layout=slider_layout)
co2_grid_slider = FloatSlider(description="CO2 Strom kg/kWh", min=0.05, max=0.6, step=0.01, value=0.33, style=slider_style, layout=slider_layout)

controls = {
    "km_total": km_slider,
    "hyb_l_per_100": hyb_l_slider,
    "hyb_kwh_per_100": hyb_kwh_slider,
    "diesel_l_per_100": diesel_slider,
    "bev_kwh_per_100": bev_slider,
    "price_petrol": price_petrol_slider,
    "price_diesel": price_diesel_slider,
    "price_kwh": price_kwh_slider,
    "co2_petrol": co2_petrol_slider,
    "co2_diesel": co2_diesel_slider,
    "co2_grid": co2_grid_slider,
}

output = Output()


def render_dashboard(**kwargs):
    df, fig = build_dashboard_state(**kwargs)
    with output:
        output.clear_output(wait=True)
        display(df)
        display(fig)
        plt.close(fig)

interactive_handle = interactive_output(render_dashboard, controls)

ui = VBox([
    heading,
    intro,
    km_slider,
    hyb_l_slider,
    hyb_kwh_slider,
    diesel_slider,
    bev_slider,
    price_petrol_slider,
    price_diesel_slider,
    price_kwh_slider,
    co2_petrol_slider,
    co2_diesel_slider,
    co2_grid_slider,
], layout=Layout(width="95%"))

display(ui, output)



Output()